In [1]:
import os
import sys
PROJECT_ROOT = '/storage/scratch1/3/grubin6/2AFC'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import os
import numpy as np
import h5py
import matplotlib.pyplot as plt


In [2]:
from notebook_tools.io import read_ops


In [3]:
list_session_data_path = [
    r'F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584',
    # '/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250811_2afc-580',
    # '/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250808_2afc-577',
    # '/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250805_2afc-568',
]

list_ops = read_ops(list_session_data_path)

In [4]:
from notebook_tools.io import create_memmap, get_memmap_path, read_masks


In [5]:
from notebook_tools.io import read_dff


In [6]:
from notebook_tools.io import read_raw_voltages


In [7]:
from notebook_tools.io import read_bpod_mat_data


In [11]:
from notebook_tools.io import remove_start_impulse, correct_vol_start, get_trigger_time, get_session_start_time, correct_time_img_center, save_trials


In [10]:
for ops in list_ops:

    print('--- Processing session ---')
    print('Processing session data in:')
    print(ops['save_path0'])
    print('-------------------------------')

    print('Reading dff traces and voltage recordings')
    dff = read_dff(ops)
    [vol_time, vol_start, vol_stim_vis, vol_img,
        vol_hifi, vol_stim_aud, vol_flir,
        vol_pmt, vol_led] = read_raw_voltages(ops)
    vol_stim_vis = remove_start_impulse(vol_time, vol_stim_vis)
    vol_stim_vis = correct_vol_start(vol_stim_vis)
    session_start_time = get_session_start_time(vol_time, vol_start)
    trial_labels = read_bpod_mat_data(ops, session_start_time, vol_time, vol_start, vol_stim_vis)
    print('Correcting 2p camera trigger time')
    # signal trigger time stamps.
    time_img, _   = get_trigger_time(vol_time, vol_img)
    # correct imaging timing.
    time_neuro = correct_time_img_center(time_img)
    # save the final data.
    print('Saving trial data')
    save_trials(
        ops, time_neuro, dff, trial_labels,
        vol_time, vol_stim_vis,
        vol_stim_aud, vol_flir,
        vol_pmt, vol_led)


--- Processing session ---
Processing session data in:
/home/ihsan/Desktop/data/Georgia_Tech/2AFC/Data/2p/YH24LG_CRBL_lobulev_20250814_2afc-584
-------------------------------
Reading dff traces and voltage recordings
Correcting 2p camera trigger time
Saving trial data


In [8]:
from notebook_tools.io import read_trial_label, zscore_normalize, read_neural_trials


In [9]:
import pandas as pd
list_neural_trials = []
for ops in list_ops:
    print('--- Processing session ---')
    print('Processing session data in:')
    print(ops['save_path0'])
    print('-------------------------------')
    print('Reading trailized neural traces with stimulus alignment')
    neural_trials = read_neural_trials(ops, 0)
    list_neural_trials.append(neural_trials)


--- Processing session ---
Processing session data in:
F:\Single_Interval_discrimination\Data_2p\YH24LG_CRBL_lobulev_20250814_2afc-584
-------------------------------
Reading trailized neural traces with stimulus alignment


# LFAD

## alginnment and data preparation

In [30]:
from notebook_tools.alignment import trim_seq, get_perception_response
from notebook_tools.lfads import prepare_data_for_lfads


[dataset_01] Extracting aligned data for state: stim_seq...


Processing Trials: 100%|██████████| 403/403 [00:00<00:00, 3874.19it/s]


Transposing data to (Trials, Time, Neurons) for LFADs compatibility...
Interpolating stimulus data...
Saving data to lfads_data.h5...
Process Complete.
Train Data Shape: (319, 180, 253) (Trials, Time, Neurons)
Valid Data Shape: (80, 180, 253)
Output saved to: f:\Single_Interval_discrimination\Code\2p\2p_2AFC_double_block_version\Test_pilot\lfads_data.h5


(array([[[ 1.78482428e-01,  2.13512397e+00,  1.18016921e-01, ...,
           3.94653976e-01, -3.62823270e-02, -5.13102829e-01],
         [ 9.11211848e-01,  1.92938864e+00,  2.11163449e+00, ...,
          -3.27257738e-02,  2.85180002e-01,  4.45467329e+00],
         [ 3.53425324e-01, -7.66439214e-02,  1.08490682e+00, ...,
           1.86676562e-01,  5.89859247e-01, -2.50495720e+00],
         ...,
         [ 6.18227780e-01, -2.03779802e-01, -6.52541459e-01, ...,
          -3.41360122e-01, -1.82480621e+00, -1.25512981e+00],
         [ 6.20932877e-01,  2.94580579e-01,  1.83105910e+00, ...,
           8.42360616e-01, -1.15232003e+00, -2.44349957e+00],
         [ 2.91585475e-01,  2.47764778e+00, -8.20709944e-01, ...,
           1.04023311e-02, -4.51547325e-01, -1.41717267e+00]],
 
        [[ 1.48153889e+00, -1.06311154e+00, -1.15000784e+00, ...,
          -1.22235286e+00, -1.03613746e+00,  3.53590325e-02],
         [ 1.06824410e+00, -2.47751474e-01, -1.23406231e+00, ...,
           1.21992707

In [32]:
# Alternative if the command above fails
import sys
!{sys.executable} -m lfads-torch.lfads_torch.run_model --config-name config --config-dir .

In [ ]:
import sys
!{sys.executable} -m lfads-torch.train --config-name config --config-dir .

f:\Single_Interval_discrimination\Code\2p\2p_2AFC_double_block_version\Test_pilot\.venv\Scripts\python.exe: No module named lfads-torch.train


: 